## Library + API keys

https://deepeval.com/integrations/models/openai?utm_source=chatgpt.com

https://deepeval.com/docs/metrics-introduction

In [1]:
!pip install langchain-unstructured unstructured langchain-text-splitters -q
!pip install langchain-openai langchain-chroma -q
!pip install deepeval rouge_score -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 8.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.8/453.8 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.6/169.6 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 22.5 MB/s eta 0:00:00
  

In [2]:
urls = [
    "https://raw.githubusercontent.com/kubernetes/website/main/content/en/docs/concepts/overview/_index.md",
    "https://raw.githubusercontent.com/kubernetes/website/main/content/en/docs/concepts/architecture/_index.md",
    "https://raw.githubusercontent.com/kubernetes/website/main/content/en/docs/concepts/containers/_index.md",
    "https://raw.githubusercontent.com/kubernetes/website/main/content/en/docs/concepts/workloads/_index.md",
    "https://raw.githubusercontent.com/kubernetes/website/main/content/en/docs/concepts/storage/_index.md",
]

In [3]:
# download NLTK data
import nltk
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [4]:
from google.colab import userdata
openai_api = userdata.get('OPENAI_API_KEY')

import os
os.environ['OPENAI_API_KEY'] = openai_api

In [5]:
# Import required functions / classes
from langchain_unstructured import UnstructuredLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma

In [6]:
# Define Models
MODEL = "gpt-5.4"
EMBEDDING_MODEL = "text-embedding-3-small"

In [7]:
# Define evaluation llm and embedding model
EVAL_MODEL = "gpt-5.4"
eval_embeddings = OpenAIEmbeddings(model = "text-embedding-3-large")

# Data + RAG

In [8]:
# Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 3000,
    chunk_overlap = 300
)

documents = []

for url in urls:
  docs =UnstructuredLoader(web_url=url).load()

  # Clean the text
  text = "\n\n".join(
      doc.page_content for doc in docs
      if doc.page_content.strip()
  )

  documents.extend(text_splitter.create_documents(
      [text], metadatas = [{"source": url}]
      )
  )

documents

[Document(metadata={'source': 'https://raw.githubusercontent.com/kubernetes/website/main/content/en/docs/concepts/overview/_index.md'}, page_content='--- reviewers: - bgrant0607 - mikedanese title: "Overview" description: > Kubernetes is a portable, extensible, open source platform for managing containerized workloads and services that facilitate both declarative configuration and automation. It has a large, rapidly growing ecosystem. Kubernetes services, support, and tools are widely available. content_type: concept weight: 20 theme_lock: light card: name: concepts weight: 10 anchors: - anchor: "#why-you-need-kubernetes-and-what-can-it-do" title: Why Kubernetes? no_list: true ---\n\n<!-- overview --> This page is an overview of Kubernetes.\n\n<!-- body -->\n\nThe name Kubernetes originates from Greek, meaning helmsman or pilot. K8s as an abbreviation results from counting the eight letters between the "K" and the "s". Google open sourced the Kubernetes project in 2014. Kubernetes comb

In [9]:
# How many documents/ chunks we have
print(f"We have {len(documents)} documents")

We have 13 documents


In [10]:
# Embed the chunks and store in Chroma
embeddings = OpenAIEmbeddings(model = EMBEDDING_MODEL)
db = Chroma.from_documents(documents, embeddings)

In [11]:
def rag(query, k = 4):
  # Build the RAG
  retrieved_docs = db.similarity_search(
      query = query, k = k)

  # Combine the context
  combined_text = "\n\n".join([doc.page_content for doc in retrieved_docs])

  # Define the prompt
  prompt = f"Based on the context {combined_text}, answer the query: {query}"

  # Call the LLM
  model = ChatOpenAI(model = MODEL)
  response_text = model.invoke(prompt)

  return response_text.content, [txt.page_content for txt in retrieved_docs]

In [12]:
# test the function
query = "What is the overview of kubernetes"
rag(query)

('Kubernetes is an open-source platform for running and managing containerized applications at scale.\n\n### In overview, Kubernetes:\n- Automates deployment of containers\n- Keeps applications running reliably\n- Scales applications up or down\n- Handles service discovery and load balancing\n- Supports storage orchestration\n- Manages rollouts and rollbacks\n- Performs self-healing when containers fail\n- Manages secrets and configuration\n- Supports batch workloads and extensibility\n\n### Why it exists\nContainers are great for packaging applications, but in production you need a system to:\n- restart failed containers\n- distribute traffic\n- place workloads efficiently across machines\n- update applications without downtime\n\nKubernetes provides that framework.\n\n### What it can do\nIt offers:\n- **Service discovery and load balancing**\n- **Storage orchestration**\n- **Automated rollouts and rollbacks**\n- **Automatic bin packing**\n- **Self-healing**\n- **Secret and configurat

# Generate Synthetic data

In [13]:
from deepeval.models import OpenAIModel
from deepeval.synthesizer import Synthesizer
import pandas as pd
import random

In [14]:
# DeepEval model
generator_model = OpenAIModel(
    model = MODEL,
    temperature = 1
)

In [15]:
# Define the target test
TESTSET_SIZE = 30

In [16]:
# How many documents times goldens products
len(documents) * 4

52

In [17]:
# Use each chunk as context
contexts = [[d.page_content] for d in documents]

# Synthetic data generator
generator = Synthesizer(
    model = generator_model
)

In [18]:
contexts[:2]

[['--- reviewers: - bgrant0607 - mikedanese title: "Overview" description: > Kubernetes is a portable, extensible, open source platform for managing containerized workloads and services that facilitate both declarative configuration and automation. It has a large, rapidly growing ecosystem. Kubernetes services, support, and tools are widely available. content_type: concept weight: 20 theme_lock: light card: name: concepts weight: 10 anchors: - anchor: "#why-you-need-kubernetes-and-what-can-it-do" title: Why Kubernetes? no_list: true ---\n\n<!-- overview --> This page is an overview of Kubernetes.\n\n<!-- body -->\n\nThe name Kubernetes originates from Greek, meaning helmsman or pilot. K8s as an abbreviation results from counting the eight letters between the "K" and the "s". Google open sourced the Kubernetes project in 2014. Kubernetes combines [over 15 years of Google\'s experience](/blog/2015/04/borg-predecessor-to-kubernetes/) running production workloads at scale with best-of-bree

In [19]:
# Generate the questions and reference answers
goldens = generator.generate_goldens_from_contexts(
    contexts = contexts,
    include_expected_output = True,
    max_goldens_per_context = 4
)
goldens[:2]

Output()

[Golden(id=None, input='How does Kubernetes use automatic bin packing with CPU/RAM requests to optimize node resource utilization?', actual_output=None, expected_output='Kubernetes uses automatic bin packing by taking the CPU and memory requests you specify for each container and scheduling those containers onto cluster nodes where they best fit. This helps Kubernetes place workloads efficiently so the available node resources are used as effectively as possible.', context=["**Automatic bin packing\n\n** You provide Kubernetes with a cluster of nodes that it can use to run containerized tasks. You tell Kubernetes how much CPU and memory (RAM) each container needs. Kubernetes can fit containers onto your nodes to make the best use of your resources.\n\n**Self-healing\n\n** Kubernetes restarts containers that fail, replaces containers, kills containers that don't respond to your user-defined health check, and doesn't advertise them to clients until they are ready to serve.\n\n**Secret an

In [20]:
for golden in goldens:
  print(golden.input)
  print(golden.expected_output)

How does Kubernetes use automatic bin packing with CPU/RAM requests to optimize node resource utilization?
Kubernetes uses automatic bin packing by taking the CPU and memory requests you specify for each container and scheduling those containers onto cluster nodes where they best fit. This helps Kubernetes place workloads efficiently so the available node resources are used as effectively as possible.
Can Kubernetes restart failed containers, kill failed health checks, and withhold traffic until ready?
Yes. Kubernetes is self-healing: it restarts failed containers, replaces them if needed, kills containers that fail user-defined health checks, and does not send client traffic to them until they are ready to serve.
Does K8s provide built-in app-level svcs like DBs, caches, or middleware, and what can run instead?
No. Kubernetes does **not** provide built-in application-level services such as **databases, caches, middleware/message buses, data-processing frameworks, or cluster storage sy

In [21]:
# Put the test data as a pandas df
test_df = pd.DataFrame([
    {
        "user_input": golden.input,
        "expected_output": golden.expected_output
    }
    for golden in goldens
])
test_df.head(2)


,user_input,expected_output
0,How does Kubernetes use automatic bin packing ...,Kubernetes uses automatic bin packing by takin...
1,"Can Kubernetes restart failed containers, kill...",Yes. Kubernetes is self-healing: it restarts f...


In [22]:
# Answer each user input
answers_gen = []
context_gen = []

for user_input in test_df['user_input']:
  answer, context = rag(user_input)
  answers_gen.append(answer)
  context_gen.append(context)

# Store it in the test_df
test_df['answer_gen'] = answers_gen
test_df['context_gen'] = context_gen

test_df.head()

,user_input,expected_output,answer_gen,context_gen
0,How does Kubernetes use automatic bin packing ...,Kubernetes uses automatic bin packing by takin...,Kubernetes uses automatic bin packing through ...,[**Automatic bin packing\n\n** You provide Kub...
1,"Can Kubernetes restart failed containers, kill...",Yes. Kubernetes is self-healing: it restarts f...,"Yes. Kubernetes provides **self-healing**, whi...",[**Automatic bin packing\n\n** You provide Kub...
2,Does K8s provide built-in app-level svcs like ...,No. Kubernetes does **not** provide built-in a...,No. Kubernetes does **not** provide built-in a...,[**Automatic bin packing\n\n** You provide Kub...
3,Why isn't Kubernetes a traditional all-inclusi...,Kubernetes isn’t a traditional all-inclusive P...,Kubernetes isn’t a traditional all-inclusive P...,[**Automatic bin packing\n\n** You provide Kub...
4,How does Cluster DNS serve svc records and aut...,Cluster DNS is a DNS server for the Kubernetes...,Cluster DNS is an in-cluster DNS server that p...,[### DNS\n\nWhile the other addons are not str...


In [23]:
test_df

,user_input,expected_output,answer_gen,context_gen
0,How does Kubernetes use automatic bin packing ...,Kubernetes uses automatic bin packing by takin...,Kubernetes uses automatic bin packing through ...,[**Automatic bin packing\n\n** You provide Kub...
1,"Can Kubernetes restart failed containers, kill...",Yes. Kubernetes is self-healing: it restarts f...,"Yes. Kubernetes provides **self-healing**, whi...",[**Automatic bin packing\n\n** You provide Kub...
2,Does K8s provide built-in app-level svcs like ...,No. Kubernetes does **not** provide built-in a...,No. Kubernetes does **not** provide built-in a...,[**Automatic bin packing\n\n** You provide Kub...
3,Why isn't Kubernetes a traditional all-inclusi...,Kubernetes isn’t a traditional all-inclusive P...,Kubernetes isn’t a traditional all-inclusive P...,[**Automatic bin packing\n\n** You provide Kub...
4,How does Cluster DNS serve svc records and aut...,Cluster DNS is a DNS server for the Kubernetes...,Cluster DNS is an in-cluster DNS server that p...,[### DNS\n\nWhile the other addons are not str...
5,Examine Kubernetes Dashboard's role in diagnos...,"Kubernetes Dashboard is a general-purpose, web...",Kubernetes Dashboard is a general-purpose web ...,[### DNS\n\nWhile the other addons are not str...
6,"How do network plugins implement CNI, allocate...",Network plugins are software components that i...,Network plugins in Kubernetes implement the **...,[### DNS\n\nWhile the other addons are not str...
7,If a prod cluster separates control-plane node...,For a production cluster with dedicated contro...,For a production cluster that uses dedicated c...,[### DNS\n\nWhile the other addons are not str...
8,"When a matching node is added, what Pod does a...","When a matching node is added, the control pla...","When a matching node is added, the DaemonSet s...",[[DaemonSet](/docs/concepts/workloads/controll...
9,"Can a CronJob sched the same Job repeatedly, u...",Yes. A **Job** runs a task **once to completio...,Yes.\n\nA **Job** runs a task **once to comple...,[[DaemonSet](/docs/concepts/workloads/controll...


In [24]:
# Sample the testsize with random seed 1502 using pandas sample with random_state
test_df_sample = test_df.sample(
    n=TESTSET_SIZE, random_state=1502).reset_index(drop=True)

# Display the first few rows
test_df_sample.head(2)

,user_input,expected_output,answer_gen,context_gen
0,"How do shared OS, image immutability, and infr...",They help ensure environmental consistency by ...,They ensure environment consistency in three c...,"[Does not dictate logging, monitoring, or aler..."
1,Examine Kubernetes Dashboard's role in diagnos...,"Kubernetes Dashboard is a general-purpose, web...",Kubernetes Dashboard is a general-purpose web ...,[### DNS\n\nWhile the other addons are not str...


# Rouge Score

In [25]:
# Import classes and libraries
from deepeval.test_case import LLMTestCase
from rouge_score import rouge_scorer
import numpy as np

In [26]:
# Rouge Score
rouge_score_list = []
scorer = rouge_scorer.RougeScorer(
    ['rougeL'],
    use_stemmer=True
)

for row in test_df_sample.to_dict(orient='records'):
  rouge_score = scorer.score(
      row['expected_output'],
      row['answer_gen']
  )['rougeL'].fmeasure
  rouge_score_list.append(rouge_score)

In [27]:
# Compute the outputs
print(rouge_score_list)
print(f"The Mean Rouge is {np.mean(rouge_score_list)}")

[0.2760942760942761, 0.1794871794871795, 0.29710144927536236, 0.2869198312236287, 0.5945945945945946, 0.1952662721893491, 0.39862542955326463, 0.6909090909090908, 0.43795620437956206, 0.20095693779904308, 0.2773722627737226, 0.25414364640883974, 0.24742268041237114, 0.2884615384615385, 0.41269841269841273, 0.3300970873786408, 0.20465116279069767, 0.2233009708737864, 0.7552447552447551, 0.30769230769230765, 0.19047619047619047, 0.2580645161290322, 0.19087136929460582, 0.19753086419753085, 0.24432809773123912, 0.4216867469879518, 0.6410256410256411, 0.609090909090909, 0.9850746268656716, 0.723404255319149]
The Mean Rouge is 0.3773516435786115


# LLM based evaluation

In [28]:
from deepeval.test_case import SingleTurnParams
from deepeval.metrics import GEval

In [45]:
correctness_metric = GEval(
    name="General Quality",
    criteria="Determine whether the actual output is good enough based on the expected output.",
    # NOTE: you can only provide either criteria or evaluation_steps, and not both
    evaluation_steps=[
        "Check whether the facts in 'actual output' are the same facts in 'expected output'",
        "Check if the general answer of 'actual output' is as a good as the 'expected output'"
    ],
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.EXPECTED_OUTPUT],
    model = EVAL_MODEL
)

In [32]:
test_df_sample.head()

,user_input,expected_output,answer_gen,context_gen
0,"How do shared OS, image immutability, and infr...",They help ensure environmental consistency by ...,They ensure environment consistency in three c...,"[Does not dictate logging, monitoring, or aler..."
1,Examine Kubernetes Dashboard's role in diagnos...,"Kubernetes Dashboard is a general-purpose, web...",Kubernetes Dashboard is a general-purpose web ...,[### DNS\n\nWhile the other addons are not str...
2,Compare roles of Kubernetes control plane vs w...,The Kubernetes control plane manages the clust...,"In Kubernetes, the **control plane** and **wor...","[--- title: ""Cluster Architecture"" weight: 30 ..."
3,Compare Deployment vs StatefulSet for Kubernet...,A **Deployment** is best for **stateless appli...,Here’s the key comparison between **Deployment...,"[--- title: ""Workloads"" weight: 55 description..."
4,"When a matching node is added, what Pod does a...","When a matching node is added, the control pla...","When a matching node is added, the DaemonSet s...",[[DaemonSet](/docs/concepts/workloads/controll...


In [30]:
# Inspect the data
for row in test_df_sample.to_dict(orient='records'):
  row_n = LLMTestCase(
      input = row['user_input'],
      expected_output = row['expected_output'],
      actual_output = row['answer_gen'])
  print(row_n)


input='How do shared OS, image immutability, and infra decoupling ensure env consistency?' actual_output='They ensure environment consistency in three complementary ways:\n\n- **Shared OS**\n  - Containers use the host OS kernel instead of shipping a full OS like a VM.\n  - This makes them **lightweight and standardized**, so the same container can run predictably across machines that support the same container runtime and kernel expectations.\n  - Because the execution model is consistent, differences between dev, test, and prod are reduced.\n\n- **Image immutability**\n  - A container image bundles the app code, runtime, libraries, and default settings into a **fixed, repeatable artifact**.\n  - Once built, you don’t modify a running container; instead, you rebuild the image and redeploy.\n  - This prevents “configuration drift” and ensures the **same image behaves the same way** in every environment.\n\n- **Infrastructure decoupling**\n  - Containers abstract the application away fr

In [34]:
 # MEasure our RAG
# Get the Simple score for each row
simple_score_list = []

for row in test_df_sample.to_dict(orient='records'):
  row_n = LLMTestCase(
      input = row['user_input'],
      expected_output = row['expected_output'],
      actual_output = row['answer_gen'])

  correctness_metric.measure(row_n)
  simple_score_list.append(correctness_metric.score)

#Display the outputs
print(simple_score_list)
print(f"The Mean Simple Score is {np.mean(simple_score_list)}")

Output()

Output()

Output()

Output()

Output()

Output()

... (21 outputs left out)

Output()

Output()

Output()

[0.9377540668798146, 0.95621765008858, 1.0, 1.0, 1.0, 0.8182425523806355, 0.9952574126822432, 1.0, 1.0, 0.9832068155891058, 1.0, 1.0, 0.9962673112655869, 1.0, 0.9320821300824609, 0.9000000000000001, 0.9519643029939322, 1.0, 0.9268941421369995, 1.0, 0.9970687769248643, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.9182425523806357, 1.0, 1.0]
The Mean Simple Score is 0.9771065904468286


# Rubrics

In [35]:
rubrics = {
    "score1_description": "The response is entirely incorrect and fails to address any aspect of the reference.",
    "score2_description": "The response contains partial accuracy but includes major errors or significant omissions that affect its relevance to the reference.",
    "score3_description": "The response is mostly accurate but lacks clarity, thoroughness, or minor details needed to fully address the reference.",
    "score4_description": "The response is accurate and clear, with only minor omissions or slight inaccuracies in addressing the reference.",
    "score5_description": "The response is completely accurate, clear, and thoroughly addresses the reference without any errors or omissions.",
}

In [36]:
from deepeval.metrics.g_eval import Rubric

In [37]:
rubrics_scorer = GEval(
    name="Rubrics Scorer",
    criteria="Evaluate how accurately and completely the actual output addresses the expected output",
    evaluation_params=[
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.EXPECTED_OUTPUT],
    rubric=[
        Rubric(score_range=(1,2), expected_outcome= rubrics['score1_description']),
        Rubric(score_range=(3,4), expected_outcome= rubrics['score2_description']),
        Rubric(score_range=(5,6), expected_outcome= rubrics['score3_description']),
        Rubric(score_range=(7,8), expected_outcome= rubrics['score4_description']),
        Rubric(score_range=(9,10), expected_outcome= rubrics['score5_description'])
    ],
    model = EVAL_MODEL
)


In [38]:
# Compute Rubrics Score
rubrics_score_list = []

for row in test_df_sample.to_dict(orient='records'):
  row_n = LLMTestCase(
      input = row['user_input'],
      expected_output = row['expected_output'],
      actual_output = row['answer_gen'])

  rubrics_scorer.measure(row_n)
  rubrics_score_list.append(rubrics_scorer.score)

# Display the outputs
print(rubrics_score_list)
print(f"The Mean Rubrics Score is {np.mean(rubrics_score_list)}")

Output()

Output()

Output()

Output()

Output()

Output()

... (21 outputs left out)

Output()

Output()

Output()

[0.8756441197753202, 0.8888888888888888, 0.8921458034168174, 0.9245357000916231, 0.8888888888888888, 0.5975045187553495, 0.9091583915340394, 1.0, 0.9308378520886826, 0.8888888888888888, 0.9867552308864312, 0.9933237055362216, 0.8888888888888891, 1.0, 0.7777777777777778, 0.8888888888888888, 0.7803308188788919, 1.0, 0.8197267409775719, 0.872439200218701, 0.9580510368002061, 0.8888888888888891, 0.894158430353063, 0.9752555401305213, 0.9915713133309728, 0.9797304973548491, 1.0, 0.8402418334317555, 0.9967430854720717, 0.9513529445428665]
The Mean Rubrics Score is 0.9093539291562356


# Factual Correctness

In [46]:
factual_correctness_metric = GEval(
    name="Factual Correctness",
    criteria="Determine whether the actual output is factually correct compared to the expected output.",
    # NOTE: you can only provide either criteria or evaluation_steps, and not both
    evaluation_steps=[
        "Check whether the facts in 'actual output' are the same facts in 'expected output'",
        "Penalize significantly wrong facts",
        "Penalize omitted facts half as much as wrong facts",
        "Penalize non-relevant facts half as much as omitted facts"
    ],
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.EXPECTED_OUTPUT],
    model = EVAL_MODEL)

In [47]:
 # MEasure our RAG
# Get the Simple score for each row
factual_correctness_list = []

for row in test_df_sample.to_dict(orient='records'):
  row_n = LLMTestCase(
      input = row['user_input'],
      expected_output = row['expected_output'],
      actual_output = row['answer_gen'])

  factual_correctness_metric.measure(row_n)
  factual_correctness_list.append(factual_correctness_metric.score)

#Display the outputs
print(factual_correctness_list)
print(f"The Mean Simple Score is {np.mean(factual_correctness_list)}")

Output()

Output()

Output()

Output()

Output()

Output()

... (21 outputs left out)

Output()

Output()

Output()

[0.9119202922022117, 0.9022977369910026, 0.9268941421369995, 1.0, 1.0, 0.7562176500885797, 0.9817574476193645, 1.0, 1.0, 0.9817574476193645, 0.9125907353531744, 1.0, 0.9, 0.9817574476193645, 0.8939913349825991, 0.8952574126822433, 0.8471149307742918, 0.9904650535100892, 0.8182425523806355, 1.0, 0.995257412682243, 0.9999999999999998, 0.9970687769248643, 0.9962673112655871, 1.0000000000000002, 0.9977022630089974, 1.0, 0.9, 1.0, 1.0]
The Mean Simple Score is 0.9528853315947202


# Semantic similarity

In [48]:
# Create a function for cosine similarity
def semantic_similarity(reference, answer):

  # Compute the embeddings
  vectors = eval_embeddings.embed_documents(
      [reference, answer]
  )
  reference_vector = np.array(vectors[0])
  answer_vector = np.array(vectors[1])


# Compute the similarity
  return np.dot(reference_vector, answer_vector) / (
      np.linalg.norm(reference_vector) * np.linalg.norm(answer_vector))


In [49]:
# Compute the similarity for each row
semantic_score_list = []
for row in test_df_sample.to_dict(orient='records'):
  semantic_score = semantic_similarity(
      row['expected_output'],
      row['answer_gen']
  )
  semantic_score_list.append(semantic_score)


# Display the outputs
print(semantic_score_list)
print(f"The Mean Semantic Score is {np.mean(semantic_score_list)}")


[np.float64(0.9001553008924913), np.float64(0.8432617382336313), np.float64(0.8783071877526909), np.float64(0.8750088584688083), np.float64(0.9221106320407749), np.float64(0.8520464192751201), np.float64(0.9352597457521107), np.float64(0.9560403613890327), np.float64(0.9027870333677239), np.float64(0.7657138903365184), np.float64(0.8571349079519579), np.float64(0.8639375551758537), np.float64(0.8165320277407121), np.float64(0.8835005178331888), np.float64(0.8202979214957319), np.float64(0.9069863315183332), np.float64(0.8132867243028408), np.float64(0.867101016549336), np.float64(0.8995651386943558), np.float64(0.8129158592469767), np.float64(0.8987298724344492), np.float64(0.8147327890304739), np.float64(0.834140096236277), np.float64(0.8951011961012054), np.float64(0.8674451834630414), np.float64(0.9469171850848389), np.float64(0.8763254596419473), np.float64(0.9142342836618613), np.float64(0.9601304736645868), np.float64(0.9390296607741373)]
The Mean Semantic Score is 0.877291178937

# Context Recall

In [51]:
from deepeval.metrics import ContextualRecallMetric

In [52]:
# Create the scorer
contextual_recall_scorer = ContextualRecallMetric(
    model = EVAL_MODEL)

In [53]:
# Compute the recall score
recall_score_list = []
for row in test_df_sample.to_dict(orient='records'):
  row = LLMTestCase(
      input = row['user_input'],
      expected_output = row['expected_output'],
      actual_output = row['answer_gen'],
      retrieval_context=row['context_gen']
      )

  contextual_recall_scorer.measure(row)
  recall_score_list.append(contextual_recall_scorer.score)

# display the results
print(recall_score_list)
print(f"The Mean Recall Score is {np.mean(recall_score_list)}")

Output()

Output()

Output()

Output()

Output()

Output()

... (21 outputs left out)

Output()

Output()

Output()

[1.0, 0.5, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.6666666666666666, 1.0, 1.0, 1.0, 1.0, 0.5, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
The Mean Recall Score is 0.9555555555555555


# Answer Relevancy

In [54]:
from deepeval.metrics import AnswerRelevancyMetric

In [55]:
# Define the scorer
answer_relevancy_scorer = AnswerRelevancyMetric(
    model = EVAL_MODEL)

In [56]:
# Compute the metric
relevancy_score_list = []
for row in test_df_sample.to_dict(orient='records'):
  row = LLMTestCase(
      input = row['user_input'],
      actual_output = row['answer_gen'],
      )

  answer_relevancy_scorer.measure(row)
  relevancy_score_list.append(answer_relevancy_scorer.score)

# Display the outcomes
print(relevancy_score_list)
print(f"The Mean Relevancy Score is {np.mean(relevancy_score_list)}")

Output()

Output()

Output()

Output()

Output()

Output()

... (21 outputs left out)

Output()

Output()

Output()

[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.8, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
The Mean Relevancy Score is 0.9933333333333334
